In [1]:
%%writefile demo_sort.cu
#include <iostream>
#include <vector>
#include <cuda_runtime.h>
#include <cstdio>


#define CUDA_CHECK(x) \
    do { \
        cudaError_t err = x; \
        if(err != cudaSuccess){ \
            fprintf(stderr, "CUDA Error: %s\n", cudaGetErrorString(err)); \
            exit(1); \
        } \
    } while(0)


// CPU

void compare_and_swap(int* arr, int i, int j, int dir){
    bool swap_needed = (arr[i] > arr[j]);
    if(dir == 0) swap_needed = !swap_needed;

    if(swap_needed){
        int tmp = arr[i];
        arr[i] = arr[j];
        arr[j] = tmp;
    }
}

void bitonic_merge(int* arr, int low, int size, int dir){
    if(size > 1){
        int k = size/2;
        for(int i = low; i < low+k; i++){
            compare_and_swap(arr, i, i+k, dir);
        }
        bitonic_merge(arr, low, k, dir);
        bitonic_merge(arr, low+k, k, dir);
    }
}

void bitonic_sort_cpu(int* arr, int low, int size, int dir){
    if(size > 1){
        int k = size/2;

        bitonic_sort_cpu(arr, low, k, 1);
        bitonic_sort_cpu(arr, low+k, k, 0);

        bitonic_merge(arr, low, size, dir);
    }
}


// GPU

__global__ void bitonic_kernel(int* data, int j, int k){

    int idx = blockIdx.x * blockDim.x + threadIdx.x;
    int partner = idx ^ j;

    if(partner > idx){
        int dir = ((idx & k) == 0);

        if( (data[idx] > data[partner]) == dir ){
            int temp = data[idx];
            data[idx] = data[partner];
            data[partner] = temp;
        }
    }
}


// GPU wrapper
void sortGPU(int* arr){

    int N = 8;
    int* d_data;

    CUDA_CHECK( cudaMalloc(&d_data, N * sizeof(int)) );
    CUDA_CHECK( cudaMemcpy(d_data, arr, N*sizeof(int), cudaMemcpyHostToDevice) );

    int threads = 8;
    int blocks = 1;

    for(int stage = 1; stage <= 3; stage++){
        int k = 1 << stage;
        for(int step = stage-1; step >= 0; step--){
            int j = 1 << step;

            bitonic_kernel<<<blocks, threads>>>(d_data, j, k);
            CUDA_CHECK( cudaDeviceSynchronize() );
        }
    }

    CUDA_CHECK( cudaMemcpy(arr, d_data, N*sizeof(int), cudaMemcpyDeviceToHost) );
    CUDA_CHECK( cudaFree(d_data) );
}




// Main

int main(){

    std::cout << "Bitonic Sort Demo (CPU/GPU)\n";
    system("nvidia-smi");

    int arr[8];

    std::cout << "\nEnter 8 numbers:\n";
    for(int i = 0; i < 8; i++){
        std::cin >> arr[i];
    }

    std::cout << "\nChoose method:\n";
    std::cout << "1 - CPU\n";
    std::cout << "2 - GPU\n";

    int choice;
    std::cin >> choice;

    std::cout << "\nBefore sorting:\n";
    for(int i = 0; i < 8; i++)
        std::cout << arr[i] << " ";
    std::cout << "\n";

    if(choice == 1){
        bitonic_sort_cpu(arr, 0, 8, 1);
        std::cout << "(CPU version used)\n";
    }
    else if(choice == 2){
        sortGPU(arr);
        std::cout << "(GPU version used)\n";
    }
    else{
        std::cout << "Invalid choice!\n";
        return 0;
    }

    std::cout << "\nAfter sorting:\n";
    for(int i = 0; i < 8; i++)
        std::cout << arr[i] << " ";
    std::cout << "\n";

    return 0;
}


Writing demo_sort.cu


In [2]:
!nvcc demo_sort.cu -o demo_sort -arch=sm_75

In [3]:
!./demo_sort

Bitonic Sort Demo (CPU/GPU)
Sun Nov 23 09:03:18 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   64C    P8             15W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-------------------